# Step 05 - Analyze Results

Read the latest run outputs and generate analysis plots/metrics (no simulation run here).

In [ ]:
from pathlib import Path
import glob
import numpy as np
import matplotlib.pyplot as plt


def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "main.py").exists() and (candidate / "al.gga.psp").exists():
            return candidate
    raise RuntimeError("Project root not found (expected main.py and al.gga.psp).")


ROOT = find_project_root(Path.cwd())
CASE_NAME = "nc_large_vac01"
CASE_DIR = ROOT / "cases" / CASE_NAME

SMOOTH_WINDOW = 7  # odd number recommended


In [ ]:
runs = sorted(glob.glob(str(CASE_DIR / "results" / f"{CASE_NAME}_*")))
assert runs, f"No runs for case: {CASE_NAME}"
latest = Path(runs[-1])
summary = latest / "summary.csv"

data = np.genfromtxt(str(summary), delimiter=",", names=True, dtype=None, encoding="utf-8")
field_names = list((data.dtype.names or []))
stress_key = "eng_stress_top_GPa" if "eng_stress_top_GPa" in field_names else "sigma_zz_GPa"

if data.shape == ():
    strain = np.array([float(data["strain"])])
    stress = np.array([float(data[stress_key])])
    max_gap = np.array([float(data["max_gap_z_A"])])
else:
    strain = np.asarray(data["strain"], dtype=float)
    stress = np.asarray(data[stress_key], dtype=float)
    max_gap = np.asarray(data["max_gap_z_A"], dtype=float)

# Convert to positive tensile axis when values are mostly negative
if np.mean(stress < 0.0) > 0.5:
    stress_t = -stress
    stress_label = f"-{stress_key} (GPa)"
else:
    stress_t = stress
    stress_label = f"{stress_key} (GPa)"

order = np.argsort(strain)
strain = strain[order]
stress_t = stress_t[order]
max_gap = max_gap[order]

# Simple moving average smoothing
w = max(1, int(SMOOTH_WINDOW))
if w % 2 == 0:
    w += 1
if w > len(stress_t):
    w = len(stress_t) if len(stress_t) % 2 == 1 else max(1, len(stress_t) - 1)

if w > 1:
    kernel = np.ones(w) / w
    stress_s = np.convolve(stress_t, kernel, mode="same")
else:
    stress_s = stress_t.copy()

i_uts = int(np.argmax(stress_t))
uts = float(stress_t[i_uts])
uts_strain = float(strain[i_uts])

print("Latest run:", latest)
print("Stress key:", stress_key)
print("UTS (from raw points):", uts)
print("UTS strain:", uts_strain)
print("Final strain:", float(strain[-1]))
print("Final max_gap_z_A:", float(max_gap[-1]))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(strain, stress_t, "o", markersize=3, alpha=0.6, label="Raw")
ax.plot(strain, stress_s, "-", linewidth=1.8, label=f"Smoothed (w={w})")
ax.scatter([uts_strain], [uts], s=60, marker="*", label="UTS")
ax.set_xlabel("Engineering strain")
ax.set_ylabel(stress_label)
ax.set_title(f"Stress-Strain ({CASE_NAME})")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()

out = latest / "stress_strain_smoothed.png"
fig.savefig(out, dpi=160)
print("Saved:", out)
plt.show()
